# Sales Trend and Time-Based Performance Analysis for Afficionado Coffee Roasters

This notebook performs the complete exploratory data analysis required for the internship project using the date-enabled coffee shop transaction dataset.

**Important data note:** The supplied date-enabled dataset contains transactions from **1 January 2023 to 30 June 2023**, not the 2025 period stated in the original project brief. The analysis therefore reports the actual dataset period and does not relabel it as 2025.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = "../data/Coffee Shop Sales New.xlsx"
df = pd.read_excel(DATA_PATH)

print("Shape:", df.shape)
print(df.head())
print(df.columns.tolist())


In [ ]:
# Data validation and type conversion
df["transaction_date"] = pd.to_datetime(df["transaction_date"], errors="coerce")
df["transaction_time"] = pd.to_datetime(
    df["transaction_time"].astype(str),
    format="%H:%M:%S",
    errors="coerce"
)

print("Date range:", df["transaction_date"].min(), "to", df["transaction_date"].max())
print("Missing values:\n", df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate transaction IDs:", df["transaction_id"].duplicated().sum())
print("Non-positive quantities:", (df["transaction_qty"] <= 0).sum())
print("Non-positive prices:", (df["unit_price"] <= 0).sum())


In [ ]:
# Feature engineering
df["revenue"] = df["transaction_qty"] * df["unit_price"]
df["hour"] = df["transaction_time"].dt.hour
df["day_of_week"] = df["transaction_date"].dt.day_name()
df["day_of_week_num"] = df["transaction_date"].dt.dayofweek
df["week_start"] = df["transaction_date"] - pd.to_timedelta(
    df["transaction_date"].dt.dayofweek, unit="D"
)
df["month"] = df["transaction_date"].dt.to_period("M").astype(str)
df["is_weekend"] = df["day_of_week_num"] >= 5

def get_bucket(hour):
    if 6 <= hour <= 11:
        return "Morning"
    elif 12 <= hour <= 16:
        return "Afternoon"
    elif 17 <= hour <= 21:
        return "Evening"
    return "Late Hours"

df["time_bucket"] = df["hour"].apply(get_bucket)

df.head()


## 1. Dataset Overview

In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Stores:", df["store_location"].nunique())
print("Product categories:", df["product_category"].nunique())
print("Product types:", df["product_type"].nunique())
print("Total revenue:", round(df["revenue"].sum(), 2))
print("Total quantity:", int(df["transaction_qty"].sum()))
print("Total transactions:", df["transaction_id"].nunique())


## 2. Daily Revenue Trends

In [ ]:
daily = df.groupby("transaction_date").agg(
    revenue=("revenue","sum"),
    transactions=("transaction_id","nunique"),
    quantity=("transaction_qty","sum")
).reset_index()

plt.figure(figsize=(14,5))
plt.plot(daily["transaction_date"], daily["revenue"])
plt.title("Daily Revenue Trend")
plt.xlabel("Date")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Highest revenue day:")
print(daily.loc[daily["revenue"].idxmax()])
print("Lowest revenue day:")
print(daily.loc[daily["revenue"].idxmin()])


## 3. Weekly Revenue and Transaction Aggregation

In [ ]:
weekly = df.groupby("week_start").agg(
    revenue=("revenue","sum"),
    transactions=("transaction_id","nunique"),
    quantity=("transaction_qty","sum")
).reset_index()

display(weekly)

plt.figure(figsize=(12,5))
plt.bar(weekly["week_start"].astype(str), weekly["revenue"])
plt.title("Weekly Revenue")
plt.xlabel("Week Starting")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 4. Day-of-Week Performance

In [ ]:
day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]

dow = df.groupby(["day_of_week_num","day_of_week"]).agg(
    revenue=("revenue","sum"),
    transactions=("transaction_id","nunique"),
    quantity=("transaction_qty","sum"),
    days_observed=("transaction_date","nunique")
).reset_index().sort_values("day_of_week_num")

dow["avg_revenue_per_day"] = dow["revenue"] / dow["days_observed"]
dow["avg_transactions_per_day"] = dow["transactions"] / dow["days_observed"]

display(dow)

plt.figure(figsize=(10,5))
plt.bar(dow["day_of_week"], dow["avg_revenue_per_day"])
plt.title("Average Revenue per Calendar Day by Day of Week")
plt.xlabel("Day")
plt.ylabel("Average Daily Revenue")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

weekday_weekend = df.assign(
    period=np.where(df["is_weekend"], "Weekend", "Weekday")
).groupby("period").agg(
    revenue=("revenue","sum"),
    transactions=("transaction_id","nunique"),
    quantity=("transaction_qty","sum")
).reset_index()

display(weekday_weekend)


## 5. Hourly Demand Analysis

In [ ]:
hourly = df.groupby("hour").agg(
    revenue=("revenue","sum"),
    transactions=("transaction_id","nunique"),
    quantity=("transaction_qty","sum")
).reset_index()

fig, ax = plt.subplots(figsize=(12,5))
ax.plot(hourly["hour"], hourly["transactions"], marker="o")
ax.set_title("Hourly Transaction Volume")
ax.set_xlabel("Hour")
ax.set_ylabel("Transactions")
ax.set_xticks(range(24))
plt.tight_layout()
plt.show()

peak_hour = hourly.loc[hourly["transactions"].idxmax()]
slow_hour = hourly.loc[hourly["transactions"].idxmin()]
print("Peak hour:", int(peak_hour["hour"]), "transactions:", int(peak_hour["transactions"]))
print("Lowest observed hour:", int(slow_hour["hour"]), "transactions:", int(slow_hour["transactions"]))


## 6. Time-Bucket Analysis

In [ ]:
bucket_order = ["Morning","Afternoon","Evening","Late Hours"]
bucket = df.groupby("time_bucket", observed=False).agg(
    revenue=("revenue","sum"),
    transactions=("transaction_id","nunique"),
    quantity=("transaction_qty","sum")
).reindex(bucket_order).reset_index()

display(bucket)

plt.figure(figsize=(9,5))
plt.bar(bucket["time_bucket"], bucket["revenue"])
plt.title("Revenue by Time Bucket")
plt.xlabel("Time Bucket")
plt.ylabel("Revenue")
plt.tight_layout()
plt.show()


## 7. Cross-Location Temporal Comparison

In [ ]:
store_hour = df.groupby(["store_location","hour"]).agg(
    revenue=("revenue","sum"),
    transactions=("transaction_id","nunique"),
    quantity=("transaction_qty","sum")
).reset_index()

plt.figure(figsize=(12,6))
for store, g in store_hour.groupby("store_location"):
    plt.plot(g["hour"], g["transactions"], marker="o", label=store)
plt.title("Hourly Transaction Volume by Store")
plt.xlabel("Hour")
plt.ylabel("Transactions")
plt.legend()
plt.tight_layout()
plt.show()

heat = df.pivot_table(
    index="store_location",
    columns="hour",
    values="transactions",
    aggfunc="sum",
    fill_value=0
)

plt.figure(figsize=(14,4))
sns.heatmap(heat, annot=True, fmt=".0f")
plt.title("Store vs Hour Transaction Heatmap")
plt.xlabel("Hour")
plt.ylabel("Store")
plt.tight_layout()
plt.show()

store_peaks = store_hour.loc[
    store_hour.groupby("store_location")["transactions"].idxmax()
].sort_values("store_location")

display(store_peaks)


## 8. Store-Level Sales Trend

In [ ]:
store_daily = df.groupby(["transaction_date","store_location"]).agg(
    revenue=("revenue","sum"),
    transactions=("transaction_id","nunique")
).reset_index()

plt.figure(figsize=(14,6))
for store, g in store_daily.groupby("store_location"):
    plt.plot(g["transaction_date"], g["revenue"], label=store)
plt.title("Daily Revenue Trend by Store")
plt.xlabel("Date")
plt.ylabel("Revenue")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 9. Product Category Context

In [ ]:
category = df.groupby("product_category").agg(
    revenue=("revenue","sum"),
    quantity=("transaction_qty","sum"),
    transactions=("transaction_id","nunique")
).sort_values("revenue", ascending=False).reset_index()

display(category)

plt.figure(figsize=(12,5))
plt.bar(category["product_category"], category["revenue"])
plt.title("Revenue by Product Category")
plt.xlabel("Product Category")
plt.ylabel("Revenue")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 10. Evidence-Based Findings and Recommendations

Use the outputs above to document:

- Overall daily and weekly revenue direction.
- Highest and lowest revenue dates.
- Busiest and slowest days of the week.
- Weekday versus weekend differences.
- Peak transaction hour and low-demand periods.
- Morning rush, midday slowdown and evening demand.
- Store-specific peak-hour alignment or divergence.
- Staffing recommendations by location and time.
- Inventory priorities before peak demand.
- Promotions for lower-demand periods.

**Important:** Recommendations must be based on the computed results above rather than assumptions.


## 11. Data Limitation and Scope Note

The date-enabled dataset used in this notebook covers **1 January 2023 to 30 June 2023**. The original project brief refers to 2025, so the final report should explicitly state the actual coverage of the supplied dataset. The analysis should not claim that these results represent the full 2025 calendar year unless a verified 2025 date-enabled dataset is provided.